# RDLNet 两阶段训练（Colab 最小可跑）

1. **挂载 Google Drive**：**数据与 checkpoint** 放在 `DRIVE_ROOT`（持久）。**代码**每次在第 2 格重新 clone 到 **`/content/doc-scan`**（本机，断连即无，不占 Drive）。  
2. **拉代码 / 装依赖**：第 2 格 **带 submodule** clone；私有库改 `REPO_URL`。  
3. **阶段 1 蒸馏** → 权重存 `DRIVE_ROOT/checkpoints`。**只有 `train_distill` 整段跑成功**才会生成 `distill_stage1.pt`；若报错/中断，Drive 里 `checkpoints` 可能仍是空的（属正常）。  
4. **阶段 2 RDLNet**：`--resume` 续跑；`--save-every-steps` 防中途断线。

把 `DRIVE_ROOT` 改成你在 Drive 里放数据与权重的目录。

In [ ]:
# @title 1. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# @title 2. 每次重新 clone 到 /content + 依赖（只改 DRIVE_ROOT、REPO_URL）
import os, sys, subprocess, shutil

_WORK = '/content'  # 删 REPO_DIR 前必须回到此目录，否则 cwd 在已删路径里 git/! 会报错
DRIVE_ROOT = '/content/drive/MyDrive'  # 数据与 checkpoint；改成你的子目录
REPO_DIR = '/content/doc-scan'  # 代码放本机，每次先删再 clone
REPO_URL = 'https://github.com/weinixuehao/rdlnet.git'  # 私有库见上方说明

os.chdir(_WORK)
os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.check_call(
    ['git', 'clone', '--recurse-submodules', REPO_URL, REPO_DIR],
    cwd=_WORK,
)
subprocess.check_call(
    ['git', '-C', REPO_DIR, 'submodule', 'update', '--init', '--recursive'],
    cwd=_WORK,
)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
req = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.isfile(req):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
seg = os.path.join(REPO_DIR, 'segment-anything')
if os.path.isdir(seg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', seg])
print('OK cwd=', os.getcwd(), '| DRIVE_ROOT=', DRIVE_ROOT)


OK cwd= /content/doc-scan | DRIVE_ROOT= /content/drive/MyDrive


In [ ]:
# @title 3a. 下载 SAM 教师权重（sam_vit_h_4b8939.pth）
import os

IMAGE_DIR = os.path.join(DRIVE_ROOT, 'images_unlabeled')
_sam_url = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
TEACHER_CKPT = os.path.join(CHECKPOINT_DIR, 'sam_vit_h_4b8939.pth') # 约 2.39 GB
!wget -c -O $TEACHER_CKPT $_sam_url

In [ ]:
# @title 3b. 阶段 1：蒸馏训练（train_distill）
import os
from pathlib import Path

REPO_DIR = '/content/doc-scan'  # 与第 2 格一致

IMAGE_DIR = os.path.join(DRIVE_ROOT, 'train2017')
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')
TEACHER_CKPT = os.path.join(CHECKPOINT_DIR, 'sam_vit_h_4b8939.pth')
OUT = os.path.join(CHECKPOINT_DIR, 'distill_stage1.pt')

_img_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError('缺少目录: ' + IMAGE_DIR)
_nimg = sum(1 for p in Path(IMAGE_DIR).rglob('*') if p.is_file() and p.suffix.lower() in _img_ext)
print('图片数:', _nimg, '|', IMAGE_DIR)
if _nimg == 0:
    raise RuntimeError('请先往该目录放入 jpg/png')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

%cd {REPO_DIR}
# 续跑可在下行加: --resume {OUT} --epochs 10
!python -u train_distill.py --image-dir {IMAGE_DIR} --teacher-checkpoint {TEACHER_CKPT} --output {OUT} --epochs 1 --batch-size 1 --img-size 1024 --num-workers 0 --save-every-steps 200

print('阶段1 期望输出 ->', OUT)

FileNotFoundError: 缺少目录: /content/drive/MyDrive/train2017_30K

In [ ]:
# @title 5. 阶段 2：RDLNet（需 annotations.json；骨干用阶段 1 权重）
import subprocess, sys, os

ANN = os.path.join(DRIVE_ROOT, 'annotations.json')
IMG_ROOT = DRIVE_ROOT  # JSON 里 file_name / mask 相对此根目录
DISTILL_CKPT = os.path.join(DRIVE_ROOT, 'checkpoints', 'distill_stage1.pt')
OUT2 = os.path.join(DRIVE_ROOT, 'checkpoints', 'rdlnet.pt')

cmd = [
    sys.executable,
    '-u',
    'train_rdlnet.py',
    '--annotations',
    ANN,
    '--image-root',
    IMG_ROOT,
    '--distill-checkpoint',
    DISTILL_CKPT,
    '--output',
    OUT2,
    '--epochs',
    '5',
    '--batch-size',
    '1',
    '--img-size',
    '1024',
    '--num-workers',
    '0',
    '--save-every-steps',
    '100',
    # 续跑：
    # '--resume', OUT2,
    # '--epochs', '20',
]
print('运行:', ' '.join(cmd))
r = subprocess.run(cmd, cwd=REPO_DIR)
if r.returncode != 0:
    raise RuntimeError('train_rdlnet 退出码 %s' % (r.returncode,))

## 断线后续跑

- **阶段 1**：把 `train_distill.py` 加上 `--resume` 指向 `distill_stage1.pt`（或 `distill_stage1_latest.pt`），`--epochs` 写「还要训几轮」。  
- **阶段 2**：`--resume` 指向 `rdlnet.pt` 或 `rdlnet_latest.pt`，**不要**再带 `--distill-checkpoint`。

显存不够时先把 `--img-size` 改为 `512`，阶段 1 与 2 必须一致。